<a href="https://colab.research.google.com/github/lythuachau/Colab-Setup/blob/main/FreeVideo_Colab_A100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 FreeVideo trên Google Colab A100

Notebook này được chuyển từ bản ComfyUI + MiniMax-H3 đang mở sang **FreeVideo chính thức**.

- Giữ cấu hình **A100 40 GB** và lưu video/model trên Google Drive khi bật tùy chọn.
- FreeVideo cần **Python 3.12**; Colab hiện dùng **Python 3.13**, vì vậy phần chạy FreeVideo dùng môi trường riêng.
- Bỏ các custom node MiniMax/TeaCache và danh sách model/LoRA cũ. Dùng bộ cài và model do FreeVideo quản lý.
- Mở giao diện qua **Colab Proxy**, có thêm `?freevideo=launch`.

Chạy lần lượt bước 1 → 4. Lần cài đầu cần tải thư viện và model lớn. Việc chấp nhận giấy phép model do anh xác nhận trước khi tải.

Nguồn chính thức: https://github.com/FlashML-org/FreeVideo


## 1. Kiểm tra GPU A100

Ô này kiểm tra GPU, driver CUDA, Python và dung lượng ổ đĩa. Không tải model ở bước này.

In [ ]:
# @title 1. Kiểm tra A100 và môi trường Colab
import sys, subprocess, shutil
from pathlib import Path
print('Python:', sys.version)
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=True).stdout)
print('Dung lượng ổ Colab (GiB):', round(shutil.disk_usage('/content').free / 2**30, 1))
COMFY = Path('/content/ComfyUI')
FREEVIDEO = COMFY / 'custom_nodes' / 'FreeVideo'


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Tue Nov  3 02:16:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             

## 2. Cài ComfyUI và môi trường FreeVideo

FreeVideo chạy bằng Python 3.12 riêng để tương thích với Colab Python 3.13. Dùng mã nguồn và thư viện FreeVideo chính thức, không cài các node MiniMax/TeaCache cũ.

In [ ]:
# @title 2. Cài ComfyUI và FreeVideo bằng Python 3.12 riêng
import os, subprocess, sys
from pathlib import Path
COMFY = Path('/content/ComfyUI')
FREEVIDEO = COMFY / 'custom_nodes' / 'FreeVideo'
ROOT = Path('/content/FreeVideo_Runtime')
VENV = Path('/content/freevideo_ui_env')
PYTHON = VENV / 'bin' / 'python'
def run(*args, cwd=None, env=None):
    print('>', ' '.join(map(str, args)), flush=True)
    p = subprocess.Popen(list(map(str, args)), cwd=str(cwd) if cwd else None,
                         env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    try:
        for line in p.stdout:
            print(line, end='', flush=True)
        if p.wait():
            raise RuntimeError(f'Lệnh thất bại (exit {p.returncode})')
    except KeyboardInterrupt:
        p.terminate()
        raise
if not (COMFY / '.git').exists():
    run('git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI.git', COMFY)
if not (FREEVIDEO / '.git').exists():
    run('git', 'clone', '--depth', '1', 'https://github.com/FlashML-org/FreeVideo.git', FREEVIDEO)
# Không dùng Python 3.13 của kernel để chạy FreeVideo.
run(sys.executable, '-m', 'pip', 'install', '-q', 'uv')
if not PYTHON.exists():
    run(sys.executable, '-m', 'uv', 'venv', '--python', '3.12', VENV)
run(sys.executable, '-m', 'uv', 'pip', 'install', '--python', PYTHON,
    '-r', COMFY / 'requirements.txt', str(FREEVIDEO) + '[runtime]')
run(PYTHON, '-c', "import sys,torch; print('Python:',sys.version); print('Torch:',torch.__version__); print('GPU:',torch.cuda.get_device_name(0)); assert torch.cuda.is_available()")
for directory in (COMFY, FREEVIDEO):
    print(directory.name, 'commit:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=directory, text=True).strip())
print('✅ Môi trường giao diện đã sẵn sàng. Model được cài ở bước 3.')


> /usr/bin/python3 -m pip install -q uv
> /usr/bin/python3 -m uv venv --python 3.12 /content/freevideo_ui_env
Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: freevideo_ui_env
Activate with: source freevideo_ui_env/bin/activate
> /usr/bin/python3 -m uv pip install --python /content/freevideo_ui_env/bin/python -r /content/ComfyUI/requirements.txt /content/ComfyUI/custom_nodes/FreeVideo[runtime]
Using Python 3.12.3 environment at: freevideo_ui_env
Resolved 115 packages in 1.25s
Prepared 114 packages in 25.97s
Installed 115 packages in 295ms
 + accelerate==1.14.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + alembic==1.20.1
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.16.0
 + attrs==26.1.0
 + av==18.1.0
 + blake3==1.0.11
 + certifi==2026.10.23
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.0
 + comfy-aimdo==0.6.0
 + comfy-angle==0.1.1
 + comfy-kitchen==0.2.40
 + com

## 3. Model và Google Drive

Bộ cài FreeVideo tự chọn model phù hợp GPU. A100 dùng định dạng int8 theo cơ chế chọn tự động hiện tại. Không tải lại danh sách model/LoRA của notebook cũ.

Mặc định chỉ kiểm tra kế hoạch cài đặt. Đọc giấy phép trong kết quả và xác nhận đồng ý trước khi bật `ACCEPT_MODEL_LICENSE`. Bật Google Drive nếu muốn giữ model/video qua các phiên; lần đầu Colab sẽ hỏi quyền kết nối Drive.

In [ ]:
# @title 3. Lưu trữ và cài model FreeVideo
USE_GOOGLE_DRIVE = False # @param {type:"boolean"}
ACCEPT_MODEL_LICENSE = True # @param {type:"boolean"}
# Chỉ bật ACCEPT_MODEL_LICENSE sau khi đọc và đồng ý giấy phép in trong kế hoạch.
import os, subprocess
from pathlib import Path
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/FreeVideo_Colab') if USE_GOOGLE_DRIVE else Path('/content/FreeVideo_Data')
MODELS = BASE / 'Models'
OUTPUTS = BASE / 'Outputs'
for directory in (MODELS, OUTPUTS, ROOT):
    directory.mkdir(parents=True, exist_ok=True)
os.environ['FREEVIDEO_HOME'] = str(ROOT)
print('Model:', MODELS)
print('Video:', OUTPUTS)
print('Google Drive:', 'đã bật' if USE_GOOGLE_DRIVE else 'chưa bật; dữ liệu /content mất khi xóa runtime')
setup_args = ['bash', str(FREEVIDEO / 'setup.sh'), '--root', str(ROOT),
              '--models', str(MODELS), '--prepared-format', 'auto',
              '--network', 'official', '--plain', '--verbose', '--no-launch']
if ACCEPT_MODEL_LICENSE:
    setup_args += ['--yes', '--accept-model-license']
else:
    setup_args += ['--plan']
    print('Chỉ kiểm tra kế hoạch; chưa chấp nhận giấy phép hoặc tải model.')
# Giữ nhật ký để xem lại nếu cài đặt bị gián đoạn.
with open('/content/freevideo_setup.log', 'a', buffering=1) as setup_log:
    p = subprocess.Popen(setup_args, cwd=str(FREEVIDEO), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            print(line, end='', flush=True)
            setup_log.write(line)
        if p.wait():
            raise RuntimeError(f'Bộ cài FreeVideo trả mã lỗi {p.returncode}; xem log phía trên.')
    except KeyboardInterrupt:
        p.terminate()
        raise
print('✅ Đã cài model.' if ACCEPT_MODEL_LICENSE else '✅ Đã kiểm tra kế hoạch. Có thể mở giao diện ở bước 4.')


Model: /content/FreeVideo_Data/Models
Video: /content/FreeVideo_Data/Outputs
Google Drive: chưa bật; dữ liệu /content mất khi xóa runtime
  > Preparing installation

╭─ FreeVideo / Linux setup plan 
│ GPU  NVIDIA A100-SXM4-40GB · SM 8.0
│ VRAM  40.00 GiB total / 39.08 GiB free
│ RAM  83.47 GiB total / 80.02 GiB available
│ Resources  Automatic · current available VRAM/RAM
│ Installation RAM budget  76.02 GiB · system growth reserve 4.00 GiB
│ Install root  /content/FreeVideo_Runtime
│ Environment  unified · text encoder exits before video models load
│ Kernel installation  Local Sage2 source build / ABI-keyed wheel cache
│ Git  Reuse detected Git
│ Connection  Auto
│ pypi  official · connected
│ models  official · 154.76 MiB/s sample
│ encoder-models  official · 212.28 MiB/s sample
│ edge-models  official · 88.24 MiB/s sample
│ vdn-models  official · 126.26 MiB/s sample
│ image-models  official · 128.22 MiB/s sample
│ github  official · 201.15 MiB/s sample
│ git  official · 0.68s Git a

## 4. Mở FreeVideo

Giao diện chính được nhúng ngay dưới ô mã. Ô khởi động đã xử lý lỗi 403 của ComfyUI khi nhận yêu cầu từ Colab và tự nạp lại server khi model hoặc đường dẫn lưu video thay đổi.

Sau khi thấy **Model: đã cài và sẵn sàng**, anh có thể nhập mô tả và tạo video trong khung FreeVideo. Đường dẫn mở tab riêng được in thêm để tiện sử dụng; nếu tab riêng không mở được, dùng giao diện nhúng.

Nếu Colab báo không tải được JavaScript/iframe, hãy lưu và làm mới notebook.

In [ ]:
# @title 4. Mở FreeVideo trong Colab
import os, subprocess, time, urllib.request, urllib.parse, json
from pathlib import Path
from google.colab import output
PORT = 8188
LOG = Path('/content/freevideo_comfy.log')
OUTPUTS = globals().get('OUTPUTS', Path('/content/FreeVideo_Data/Outputs'))
OUTPUTS.mkdir(parents=True, exist_ok=True)
# Lấy đúng origin của proxy Colab hiện tại; không dùng wildcard CORS.
proxy_url = output.eval_js(f'google.colab.kernel.proxyPort({PORT})')
parsed = urllib.parse.urlsplit(proxy_url)
if parsed.scheme != 'https' or not parsed.hostname or not parsed.hostname.endswith(('.colab.dev', '.googleusercontent.com')):
    raise RuntimeError('Không nhận được địa chỉ proxy HTTPS hợp lệ của Colab.')
COLAB_ORIGIN = f'{parsed.scheme}://{parsed.netloc}'
launch_url = proxy_url.rstrip('/') + '/?freevideo=launch'
server_env = os.environ.copy()
server_env['FREEVIDEO_HOME'] = str(ROOT)
server_env['PYTHONUNBUFFERED'] = '1'
machine_path = ROOT / 'machine.json'
model_stamp = machine_path.stat().st_mtime_ns if machine_path.exists() else None
server_config = (model_stamp, COLAB_ORIGIN, str(OUTPUTS))
if 'comfy_proc' in globals() and comfy_proc.poll() is None and server_config != globals().get('_freevideo_server_config'):
    print('Nạp lại server với model và địa chỉ proxy Colab hiện tại.', flush=True)
    comfy_proc.terminate()
    comfy_proc.wait(timeout=20)
if 'comfy_proc' in globals() and comfy_proc.poll() is None:
    print('ComfyUI đang chạy, PID:', comfy_proc.pid)
else:
    log_handle = open(LOG, 'a', buffering=1)
    comfy_proc = subprocess.Popen([str(PYTHON), 'main.py', '--listen', '127.0.0.1',
        '--port', str(PORT), '--output-directory', str(OUTPUTS),
        '--enable-cors-header', COLAB_ORIGIN], cwd=str(COMFY), env=server_env,
        stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True)
    log_handle.close()
    print('Đã khởi động server, PID:', comfy_proc.pid, flush=True)
_freevideo_server_config = server_config
ready = False
for _ in range(90):
    if comfy_proc.poll() is not None:
        break
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/system_stats', timeout=2) as response:
            stats = json.load(response)
            ready = response.status == 200
        if ready:
            break
    except Exception:
        time.sleep(2)
if not ready:
    print(LOG.read_text(errors='replace')[-14000:] if LOG.exists() else 'Chưa có log.')
    raise RuntimeError('Server chưa sẵn sàng; xem log phía trên.')
with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/object_info', timeout=15) as response:
    nodes = json.load(response)
freevideo_nodes = [name for name in nodes if 'freevideo' in name.lower()]
if not freevideo_nodes:
    print(LOG.read_text(errors='replace')[-14000:])
    raise RuntimeError('FreeVideo chưa nạp; xem lỗi import trong log.')
# Kiểm tra tình huống trước đây gây 403 khi mở iframe từ Colab.
request = urllib.request.Request(f'http://127.0.0.1:{PORT}/', headers={
    'Sec-Fetch-Site': 'cross-site', 'Origin': COLAB_ORIGIN})
with urllib.request.urlopen(request, timeout=5) as response:
    print('Kiểm tra truy cập qua Colab:', response.status)
machine_ready = machine_path.exists() and json.loads(machine_path.read_text()).get('ready', False)
print('✅ Server và', len(freevideo_nodes), 'node FreeVideo đã nạp thành công.')
print('Model:', 'đã cài và sẵn sàng' if machine_ready else 'chưa cài; hoàn tất bước 3')
print('GPU:', stats.get('devices', [])[0].get('name', 'không xác định'))
print('Origin Colab được cho phép:', COLAB_ORIGIN)
print('🔗 FreeVideo:', launch_url)
print('Giao diện chính nằm trong khung bên dưới:')
output.serve_kernel_port_as_iframe(PORT, path='/?freevideo=launch', height=850)
print('Video lưu tại:', OUTPUTS)
print('Log:', LOG)


Nạp lại server với model và địa chỉ proxy Colab hiện tại.
Đã khởi động server, PID: 17559
Kiểm tra truy cập qua Colab: 200
✅ Server và 6 node FreeVideo đã nạp thành công.
Model: đã cài và sẵn sàng
GPU: cuda:0 NVIDIA A100-SXM4-40GB : cudaMallocAsync
Origin Colab được cho phép: https://8188-gpu-a100-s-ass1c0-269u5xymgpduh-c.asia-southeast1.prod.r.colab.dev
🔗 FreeVideo: https://8188-gpu-a100-s-ass1c0-269u5xymgpduh-c.asia-southeast1.prod.r.colab.dev/?freevideo=launch
Giao diện chính nằm trong khung bên dưới:


<IPython.core.display.Javascript object>

Video lưu tại: /content/FreeVideo_Data/Outputs
Log: /content/freevideo_comfy.log


## Dừng khi sử dụng xong

Để ngừng dùng GPU/CU, chọn **Thời gian chạy → Ngắt kết nối và xóa thời gian chạy**. Dữ liệu trên Drive được giữ lại; dữ liệu chỉ lưu trong `/content` sẽ mất.

Để kiểm tra lỗi, chạy ô khởi động lần nữa hoặc đọc `/content/freevideo_comfy.log` và `/content/freevideo_setup.log`.